In [1]:
import re
import math
import os
import pandas as pd
import numpy as np

# =============================================================================
# 1. Data Parser
# =============================================================================
def parse_dat_file(dat_filepath):
    params = {
        'scaleLength': 648.0,
        'stringSpacing': 10.8,
        'timePerMm': 0.25,
        's_max': 6, 'f_max': 22, 'p_max': 22,
        'fingerLengths_mm': [70.0, 75.0, 70.0, 50.0],
        'fingerDifficulty': [1.0, 1.5, 2.0, 3.0],
        'natural_min_spread_mm': [20.0, 40.0, 55.0],
        'availableTime': [],
        'neck_thickness': 1,
        'stringAction12thMm': 2.5,
        'd_PC': 1.5, 'd_SC': 1.0, 'd_ST': 1.0, 'd_DH': 1.0, 'd_NO': 10.0,
        'alpha_PC': 0.2, 'alpha_SC': 0.2, 'alpha_ST': 0.2, 'alpha_DH': 0.2, 'alpha_NO': 0.2
    }

    if not os.path.exists(dat_filepath):
        print("WARNING: .dat file not found. Using defaults.")
        return params

    try:
        with open(dat_filepath, 'r', encoding='utf-8-sig') as f:
            content = f.read()
    except:
        try:
            with open(dat_filepath, 'r', encoding='latin-1') as f:
                content = f.read()
        except:
            return params

    def get_arr(name):
        pat = re.compile(rf'{name}\s*=\s*\[(.*?)\];?', re.DOTALL)
        m = pat.search(content)
        if m:
            txt = re.sub(r'//.*', '', m.group(1)).replace('\n', ' ')
            return [float(v) for v in txt.split(',') if v.strip()]
        return None

    for k in params.keys():
        if isinstance(params[k], list):
            val = get_arr(k)
            if val: params[k] = val
        else:
            pat = re.compile(rf'{k}\s*=\s*([-+]?[0-9]*\.?[0-9]+);?')
            m = pat.search(content)
            if m: params[k] = float(m.group(1))
    
    params['s_max'] = int(params['s_max'])
    params['f_max'] = int(params['f_max'])
    params['neck_thickness'] = int(params.get('neck_thickness', 1))
    return params

def parse_solution_file(solution_filepath):
    notes_data = []
    if not os.path.exists(solution_filepath):
        return []
    
    try:
        with open(solution_filepath, 'r', encoding='utf-8-sig') as f:
            lines = f.readlines()
    except:
        with open(solution_filepath, 'r', encoding='latin-1') as f:
            lines = f.readlines()

    header_found = False
    data_start_idx = 0
    for i, line in enumerate(lines):
        if "Note" in line and "String" in line:
            header_found = True
            data_start_idx = i + 1
            break
    
    for i in range(data_start_idx, len(lines)):
        line = lines[i].strip()
        if not line or line.startswith('---'): continue
        parts = re.split(r'\s+', line)
        if parts[0] == "Note": continue
        
        if len(parts) >= 5:
            try:
                if parts[0] == 'R':
                        note = {'note': 'R', 'string': 0, 'fret': 0, 'hand': 0, 'finger': 0, 'eff_angle': 0.0}
                else:
                    note = {
                        'note': parts[0],
                        'string': int(parts[1]),
                        'fret': int(parts[2]),
                        'hand': int(parts[3]),
                        'finger': 0 if parts[4] == 'x' else int(parts[4]),
                        'eff_angle': float(parts[5]) if len(parts) > 5 else 0.0
                    }
                notes_data.append(note)
            except ValueError: continue
    return notes_data

# =============================================================================
# 2. Optimization Logic
# =============================================================================
class GuitarAngleOptimizer:
    def __init__(self, params):
        self.p = params
        self.fingerLengths = [0.0] + self.p['fingerLengths_mm']
        self.knuckleDist = [0.0, 0.0] + self.p['natural_min_spread_mm']
        self.reach_coeffs = [0.0, 1.0, 0.95, 0.90, 1.0]
        self.finger_mult = [0.0, 0.3, 0.3, 0.8, 0.9]
        
        neck_thickness = self.p.get('neck_thickness', 1)
        self.max_reduction_at_p16 = {1: 0.15, 2: 0.20, 3: 0.25}.get(neck_thickness, 0.15)
        self.curve_exponent = 1.0
        if self.max_reduction_at_p16 > 0.03:
            self.curve_exponent = math.log(0.03 / self.max_reduction_at_p16) / math.log(12.0 / 16.0)

        self.deg_to_rad = math.pi / 180.0
        self.rad_to_deg = 180.0 / math.pi
        sl = self.p['scaleLength']
        self.fret_pos = [sl - (sl / (2 ** (f / 12.0))) for f in range(self.p['f_max'] + 1)]
        self.eps = 0.01

    def get_low_limit(self, finger_idx):
        return {1: 15.0, 2: 15.0, 3: 20.0, 4: 25.0}.get(finger_idx, 15.0)

    def calculate_st_parameters(self, hand_pos, finger_idx, string, fret):
        """Calc Safe Zones, Slopes, and Hard Limits for a specific note."""
        res = {'BaseCost': 100000.0, 'SafeMin': 0, 'SafeMax': 0, 'SlopeL': 0, 'SlopeR': 0, 'HardMin': 0, 'HardMax': 0}
        
        if fret < hand_pos or fret > self.p['f_max']: return res
        if finger_idx not in [1, 2, 3, 4]: return res # Open string or invalid

        pivotX = self.fret_pos[hand_pos]
        pivotY = 0.0
        costs = {}
        minStCost = 100000.0
        valid_min_ang = 10.0
        valid_max_ang = -100.0
        
        # Neck factor
        ratio = min(1.0, hand_pos / 16.0) if hand_pos >= 0 else 0
        reduction = self.max_reduction_at_p16 * (ratio ** self.curve_exponent) * self.finger_mult[finger_idx]
        neckFactor = max(0.1, 1.0 - reduction)
        maxReach = self.fingerLengths[finger_idx] * self.reach_coeffs[finger_idx] * neckFactor

        for ang in range(-90, 1):
            thetaRad = ang * self.deg_to_rad
            d = self.knuckleDist[finger_idx]
            kX = pivotX + (d * math.cos(thetaRad))
            kY = pivotY + (d * math.sin(thetaRad))
            tX = self.fret_pos[fret]
            tY = (string - 1) * self.p['stringSpacing']
            dist = math.sqrt((tX - kX)**2 + (tY - kY)**2)
            
            u_dist = 999.0
            if maxReach > 0.001: u_dist = dist / maxReach
            if u_dist > 1.0: continue
            
            # Geometry
            nX, nY = -math.sin(thetaRad), math.cos(thetaRad)
            dotProd = (tX - kX) * nX + (tY - kY) * nY
            cosDelta = max(-1.0, min(1.0, (dotProd / dist) if dist > 0.001 else 1.0))
            deltaAbsDeg = math.acos(cosDelta) * self.rad_to_deg
            isLowFretSide = ((nX * (tY - kY) - nY * (tX - kX)) > 0)
            
            limit_base = self.get_low_limit(finger_idx) if isLowFretSide else 30.0
            relax_angle = 0.0
            if u_dist < 0.5: relax_angle = 10.0 if isLowFretSide else 5.0
            
            final_limit = limit_base + relax_angle
            u_angle = 999.0
            if final_limit > 0.0: u_angle = deltaAbsDeg / final_limit
            if u_angle > 1.0: continue
            
            # Cost
            c_dist = u_dist / (1.0 - u_dist + self.eps)
            dir_coeff = 1.5 if isLowFretSide else 1.0
            c_angle = dir_coeff * u_angle / (1.0 - u_angle + self.eps)
            cost = c_dist + c_angle
            
            costs[ang] = cost
            if cost < minStCost: minStCost = cost
            if ang < valid_min_ang: valid_min_ang = ang
            if ang > valid_max_ang: valid_max_ang = ang
            
        if minStCost < 50000.0:
            res['BaseCost'] = minStCost
            res['HardMin'] = valid_min_ang
            res['HardMax'] = valid_max_ang
            
            # Safe Zone Calc
            threshold = minStCost + 2.0
            safe_min, safe_max = 10, -100
            for ang in range(valid_min_ang, valid_max_ang + 1):
                if ang in costs and costs[ang] <= threshold:
                    if safe_min > ang: safe_min = ang
                    if safe_max < ang: safe_max = ang
            
            if safe_min > safe_max: safe_min, safe_max = valid_min_ang, valid_max_ang # Fallback
            
            res['SafeMin'], res['SafeMax'] = safe_min, safe_max
            
            # Slopes
            if safe_min > valid_min_ang:
                res['SlopeL'] = (costs[valid_min_ang] - costs[safe_min]) / (safe_min - valid_min_ang)
            else: res['SlopeL'] = 10.0
                
            if safe_max < valid_max_ang:
                res['SlopeR'] = (costs[valid_max_ang] - costs[safe_max]) / (valid_max_ang - safe_max)
            else: res['SlopeR'] = 10.0
            
        return res

    def optimize_effective_angles(self, notes_data):
        """DPを用いて最適なEffective Angleを計算し、notes_dataを更新する"""
        print("Starting DP optimization for Effective Angles...")
        
        # 1. 準備: 有効な音符（Rest以外）の抽出とTimePenalty計算
        valid_indices = [i for i, n in enumerate(notes_data) if n['note'] != 'R']
        n_steps = len(valid_indices)
        if n_steps == 0: return notes_data
        
        avail_time = self.p['availableTime']
        if len(avail_time) < len(notes_data):
             avail_time += [30.0] * (len(notes_data) - len(avail_time))
        
        # Time Penalty Factorsの計算
        time_penalties = {} # map note_index -> factor
        acc_time_map = {}
        for i in range(len(notes_data)):
            if notes_data[i]['note'] != 'R':
                p = i - 1
                t_sum = 0.0
                while p >= 0 and notes_data[p]['note'] == 'R':
                    t_sum += avail_time[p]
                    p -= 1
                
                acc_time = t_sum + avail_time[p] if p >= 0 else 10000.0
                acc_time_map[i] = acc_time
                if p == i or p < 0:
                    time_penalties[i] = 0.0
                else:
                    t_val = acc_time + 0.01
                    time_penalties[i] = 0.15 + (60.0 / t_val) ** 1.5
            else:
                time_penalties[i] = 0.0

        # 2. 制約パラメータの計算
        step_constraints = []
        for idx in valid_indices:
            n = notes_data[idx]
            if n['fret'] == 0: # Open String: Angle固定 (0)
                c = {'Fixed': True, 'Val': 0, 'BaseCost': 0}
            else:
                c = self.calculate_st_parameters(n['hand'], n['finger'], n['string'], n['fret'])
                c['Fixed'] = False
            step_constraints.append(c)

        # 3. DP Table 初期化
        # Angle range: -90 to 0 (index 0 to 90)
        dp = np.full((n_steps, 91), np.inf)
        parent = np.full((n_steps, 91), -1, dtype=int)
        
        alpha_ST = self.p['alpha_ST'] * self.p['d_ST']
        alpha_SC = self.p['alpha_SC'] * self.p['d_SC']

        # First step initialization
        c0 = step_constraints[0]
        if c0['Fixed']:
            range_angles = [c0['Val']]
        else:
            range_angles = range(c0['HardMin'], c0['HardMax'] + 1)
            
        for ang in range_angles:
            idx_ang = ang + 90
            st_cost = 0.0
            if not c0['Fixed']:
                if ang < c0['SafeMin']: st_cost = c0['SlopeL'] * (c0['SafeMin'] - ang)
                elif ang > c0['SafeMax']: st_cost = c0['SlopeR'] * (ang - c0['SafeMax'])
                st_cost += c0['BaseCost']
            dp[0][idx_ang] = alpha_ST * st_cost

        # 4. DP Forward Loop
        for k in range(1, n_steps):
            curr_c = step_constraints[k]
            prev_c = step_constraints[k-1]
            note_idx = valid_indices[k]
            tp = time_penalties.get(note_idx, 1.0)
            sc_weight = alpha_SC * tp

            # Current angle range
            if curr_c['Fixed']: curr_range = [curr_c['Val']]
            else: curr_range = range(curr_c['HardMin'], curr_c['HardMax'] + 1)
            
            # Previous angle range (optimization: iterate only valid DP states)
            if prev_c['Fixed']: prev_range = [prev_c['Val']]
            else: prev_range = range(prev_c['HardMin'], prev_c['HardMax'] + 1)
            
            for curr_ang in curr_range:
                curr_idx = curr_ang + 90
                
                # ST Cost Calculation
                st_cost = 0.0
                if not curr_c['Fixed']:
                    if curr_ang < curr_c['SafeMin']: st_cost = curr_c['SlopeL'] * (curr_c['SafeMin'] - curr_ang)
                    elif curr_ang > curr_c['SafeMax']: st_cost = curr_c['SlopeR'] * (curr_ang - curr_c['SafeMax'])
                    st_cost += curr_c['BaseCost']
                
                local_cost = alpha_ST * st_cost
                
                # Find best prev
                best_val = np.inf
                best_prev_ang = -1
                
                for prev_ang in prev_range:
                    prev_idx = prev_ang + 90
                    if dp[k-1][prev_idx] == np.inf: continue
                    
                    trans_cost = sc_weight * abs(curr_ang - prev_ang)
                    total = dp[k-1][prev_idx] + trans_cost
                    if total < best_val:
                        best_val = total
                        best_prev_ang = prev_ang
                
                if best_val != np.inf:
                    dp[k][curr_idx] = best_val + local_cost
                    parent[k][curr_idx] = best_prev_ang

        # 5. Backtracking
        last_step = n_steps - 1
        last_c = step_constraints[last_step]
        if last_c['Fixed']: last_range = [last_c['Val']]
        else: last_range = range(last_c['HardMin'], last_c['HardMax'] + 1)
        
        min_final_cost = np.inf
        best_end_ang = -100
        
        for ang in last_range:
            idx = ang + 90
            if dp[last_step][idx] < min_final_cost:
                min_final_cost = dp[last_step][idx]
                best_end_ang = ang
        
        if best_end_ang == -100:
            print("ERROR: Could not find feasible path.")
            return notes_data

        # Trace back
        opt_angles_map = {}
        curr_trace_ang = best_end_ang
        for k in range(last_step, -1, -1):
            note_idx = valid_indices[k]
            opt_angles_map[note_idx] = float(curr_trace_ang)
            curr_trace_ang = parent[k][curr_trace_ang + 90]
            
        # Update Data
        count_updated = 0
        for i in range(len(notes_data)):
            if i in opt_angles_map:
                notes_data[i]['eff_angle'] = opt_angles_map[i]
                count_updated += 1
            else:
                notes_data[i]['eff_angle'] = 0.0 # Rests or unreachables
        
        print(f"Optimization complete. Updated {count_updated} notes.")
        return notes_data

# =============================================================================
# Execution
# =============================================================================
if __name__ == "__main__":
    dat_file = "datas/complex_solo_mod.dat"
    sol_file = "complex_opt.txt" # 入力ファイル
    out_file = "complex_opt_optimized.txt" # 出力ファイル
    
    if os.path.exists(dat_file) and os.path.exists(sol_file):
        print("Loading parameters and data...")
        params = parse_dat_file(dat_file)
        notes = parse_solution_file(sol_file)
        
        if notes:
            optimizer = GuitarAngleOptimizer(params)
            
            # 最適化実行
            optimized_notes = optimizer.optimize_effective_angles(notes)
            
            # 結果の保存
            print(f"Saving optimized data to {out_file}...")
            with open(out_file, 'w') as f:
                f.write("Note\tString\tFret\tHandposition\tFinger\tEffAngle\n")
                for n in optimized_notes:
                    if n['note'] == 'R':
                        f.write("R\t0\t0\t0\tx\t0.0\n")
                    else:
                        finger_str = str(n['finger']) if n['finger'] != 0 else 'x'
                        f.write(f"{n['note']}\t{n['string']}\t{n['fret']}\t{n['hand']}\t{finger_str}\t{n['eff_angle']:.2f}\n")
            
            print("Done.")
        else:
            print("Error: No notes found in solution file.")
    else:
        print("Error: Input files not found.")

Loading parameters and data...
Starting DP optimization for Effective Angles...
Optimization complete. Updated 104 notes.
Saving optimized data to complex_opt_optimized.txt...
Done.
